In [ ]:
import sys
import os
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
import joblib
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.interpolate import interp1d
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, '/depot/cms/top/jpittard/Top_MLEFT/EFT_ML/')
import EFT_param_classifier as EFT_pc

In [17]:
from evaluator import EFTReweighter #existing class

#-- Config --
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
wc_dim = 16
theta1 = [0.0] * wc_dim # SM Hypothesis

VARS = ["gen_ll_cHel", "gen_ttbar_mass", "gen_c_kk", "gen_c_rr", "gen_c_nn"] #More can be added
n_vars = len(VARS)

WC_NAMES = ['ctGRe', 'ctGIm', 'cQj18', 'cQj38', 'cQj11', 'cQj31',
            'ctu8',  'ctd8',  'ctj8',  'cQu8',  'cQd8',  'ctu1',
            'ctd1',  'ctj1',  'cQu1',  'cQd1']

# WC to scan (index 0 = ctGRe)
SCAN_WC_IDX  = 0
SCAN_WC_NAME = WC_NAMES[SCAN_WC_IDX]
SCAN_RANGE   = (-10.0,10.0)
SCAN_N_PTS   = 81

# Training hyperparameters
N_THETA_TRAIN = 7      #  Number of theta points for thraining
M_EVENTS      = 8000   #  events per theta class
EPOCHS        = 20

In [18]:
# Data Path --------------------------------------------------------------
mass_regions    = ["0to700", "700to900", "900toInf"]
cross_sections  = {"0to700": 65.09, "700to900": 8.295, "900toInf": 14.03}
directory       = "/eos/purdue/store/user/lingqian/fullrun2_eft_minitrees"
struct_dir      = "/depot/cms/top/he614/notebooks/EFT_FullRun2/"
eras            = ["2016preVFP"]
channels        = ["ee", "emu", "mumu"]

USE_SAVED = False

# Reweighter -------------------------------------------------------------
print("\n[0] Loading EFT reweighter ...")
rw = EFTReweighter(
    directory_path = directory, eras = eras, channels = channels,
    mass_regions = mass_regions, cross_sections = cross_sections,
    struct_const_dir = struct_dir, step = 0
)
rw.load_structure_constants()
rw.load_observables()



[0] Loading EFT reweighter ...
[INFO] 0to700 (2016preVFP): σ=65.090 pb, L=19500 pb⁻¹, Σw=4.302e+07 → scale=2.950e-02
[INFO] 700to900 (2016preVFP): σ=8.295 pb, L=19500 pb⁻¹, Σw=1.187e+07 → scale=1.363e-02
[INFO] 900toInf (2016preVFP): σ=14.030 pb, L=19500 pb⁻¹, Σw=1.774e+07 → scale=1.542e-02


Loading SC step0: 100%|██████████| 180/180 [00:12<00:00, 14.99it/s]


Loaded 180 SC files, combined shape=(10805081, 153)


Loading observables: 100%|██████████| 180/180 [02:37<00:00,  1.14file/s]


[INFO] Total events before selection: 10805081
[INFO] Total events after selection:  10805081
[INFO] Kept fraction: 100.00%


In [20]:
print(dir(EFT_param_classifier))


['CRITICAL_VALUE_68', 'CRITICAL_VALUE_95', 'CalibratedParametric', 'DataLoader', 'IsotonicRegression', 'ParametricClassifier', 'SEED', 'StandardScaler', 'TensorDataset', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'build_input_at_theta', 'calibration_closure_plot', 'compute_all_limits', 'compute_test_statistic', 'extract_limits', 'generate_data', 'interp1d', 'joblib', 'load_artifacts', 'nn', 'norm', 'np', 'optim', 'os', 'plot_2d_limit_contour', 'plot_log_r_distribution', 'plot_test_statistic_scan', 'plot_training_history', 'plt', 'print_limits', 'roc_curve_plot', 'save_artifacts', 'scan_test_statistic', 'torch', 'tqdm', 'train_model', 'train_test_split', 'warnings']


In [23]:
# Training data ----------------------------------------------------------
if not USE_SAVED:
    print("\n[1] Building training dataset ...")
    X_batches, Y_batches, W_batches = [], [], []
    for X, Y, th0, W in EFT_param_classifier.generate_data(
        rw, VARS, wc_dim, 
        N = N_THETA_TRAIN, M = M_EVENTS,
        theta1 = theta1,
        wc_scan_index = SCAN_WC_IDX,
        wc_range = SCAN_RANGE):

        X_batches.append(X)
        Y_batches.append(Y)
        W_batches.append(W)

    X_all = np.vstack(X_batches)
    Y_all = np.concatenate(Y_batches)
    W_all = np.concatenate(W_batches)

    print(f" Total traning events: {len(Y_all):,}")

    # Train  -----------------------------------------------------------------
    print("\n[2] Training parametric classifier ...")
    model, scaler, splits, history = EFT_param_classifier.train_model(
        X_all, Y_all, device, epochs = EPOCHS)
    
    EFT_param_classifier.plot_training_history(history)

    # Calibration ------------------------------------------------------------
        # Use a held-out calibration set (separate from training if possible)
        # Here we reuse part of the training data for illustration.
        # In production: generate a fresh set of events for calibration.
    print("\n[3] Fitting weighted isotonic calibration ...")
    calibrator = EFT_param_classifier.CalibratedParametric(model, scaler, device)
    calibrator.fit_calibration(X_all, Y_all, W_all)

    # Save -------------------------------------------------------------------
    EFT_param_classifier.save_artifacts(model, scaler, calibrator)

else: 
    print("\n [1-3] Loading saved artifacts ...")
    model, scaler, calibrator = EFT_param_classifier.load_artifacts(
        inpuut_dim = n_vars + wc_dim, device = device)
    
    splits = None


[1] Building training dataset ...
[INFO] Resampled 8000 events (step=0) with EFT-weighted probabilities.
[INFO] Resampled 8000 events (step=0) with EFT-weighted probabilities.
[INFO] Resampled 8000 events (step=0) with EFT-weighted probabilities.
[INFO] Resampled 8000 events (step=0) with EFT-weighted probabilities.
[INFO] Resampled 8000 events (step=0) with EFT-weighted probabilities.
[INFO] Resampled 8000 events (step=0) with EFT-weighted probabilities.
[INFO] Resampled 8000 events (step=0) with EFT-weighted probabilities.
[INFO] Resampled 8000 events (step=0) with EFT-weighted probabilities.
[INFO] Resampled 8000 events (step=0) with EFT-weighted probabilities.
[INFO] Resampled 8000 events (step=0) with EFT-weighted probabilities.
[INFO] Resampled 8000 events (step=0) with EFT-weighted probabilities.
[INFO] Resampled 8000 events (step=0) with EFT-weighted probabilities.
[INFO] Resampled 8000 events (step=0) with EFT-weighted probabilities.
[INFO] Resampled 8000 events (step=0) with

In [55]:
import importlib
importlib.reload(EFT_param_classifier)

<module 'EFT_param_classifier' from '/depot/cms/top/jpittard/Top_MLEFT/EFT_ML/EFT_param_classifier.py'>

In [32]:
# Diagnostics ----------------------------------------------------------------
print("\n[4]Diagnostics ...")
if splits is not None:
    X_test, Y_test = splits["test"]
    EFT_param_classifier.calibration_closure_plot(calibrator, X_test, Y_test,
                                process_name=SCAN_WC_NAME)
    EFT_param_classifier.roc_curve_plot(calibrator, X_test, Y_test)


[4]Diagnostics ...
Saved artificats/plots/calibration_closure_ctGRe.pdf
Saved artifacts/plots/roc_curve.pdf


In [38]:
# Generate SM and EFT samples for test statistic _____________________________
print("\n [5] Generating SM and EFT evaluation samples ...")
sm_obs  = rw.resample_observables(theta1, max_events=200_000)
X_sm    = np.stack([np.asarray(sm_obs[k]) for k in VARS], axis = 1)
TH_sm   = np.tile(np.zeros(wc_dim), (len(X_sm), 1))
X_sm    = np.hstack([X_sm, TH_sm]).astype(np.float32)

theta_eft_demo = np.zeros(wc_dim)
theta_eft_demo[SCAN_WC_IDX] = 2.0
eft_obs = rw.resample_observables(theta_eft_demo.tolist(), max_events = 200_000)
X_eft   = np.stack([np.asarray(eft_obs[k]) for k in VARS], axis = 1)
TH_eft  = np.tile(theta_eft_demo, (len(X_eft), 1))
X_eft   = np.hstack([X_eft, TH_eft]).astype(np.float32)

EFT_param_classifier.plot_log_r_distribution(
    calibrator, X_sm, X_eft,
    theta_eft=theta_eft_demo,
    n_vars=n_vars,
    wc_name=SCAN_WC_NAME,
    wc_val=theta_eft_demo[SCAN_WC_IDX])


 [5] Generating SM and EFT evaluation samples ...
[INFO] Resampled 200000 events (step=0) with EFT-weighted probabilities.
[INFO] Resampled 200000 events (step=0) with EFT-weighted probabilities.
  Saved artifacts/plots/logr_distribution_ctGRe.pdf


In [58]:
# Test statistic scan --------------------------------------------------------
print(f"\n[6] Scanning test statistic over {SCAN_WC_NAME} ...")
theta_scan = np.linspace(SCAN_RANGE[0], SCAN_RANGE[1], SCAN_N_PTS)

scan_results = EFT_param_classifier.scan_test_statistic(
    calibrator,
    X_sm_events=X_sm,    # Asimov (expected): use SM events
    X_obs_events=X_sm,   # Replace with real observed data when unblinded
    theta_scan=theta_scan,
    wc_scan_index=SCAN_WC_IDX,
    wc_dim=wc_dim,
    n_vars=n_vars,
)

# Extract limits -------------------------------------------------------------
print("\n[7] Computing limits ...")
limits = EFT_param_classifier.compute_all_limits(scan_results, wc_name=SCAN_WC_NAME)
EFT_param_classifier.print_limits(limits)

# plots -----------------------------------------------------------------------
print("\n[8] Making limit plots ...")
EFT_param_classifier.plot_test_statistic_scan(scan_results, limits, wc_name=SCAN_WC_NAME)

EFT_param_classifier.plot_2d_limit_contour(
        calibrator, X_sm, n_vars, wc_dim,
        wc_indices=(0, 8),
        wc_names=(WC_NAMES[0], WC_NAMES[8]))

print("\n Finished script, All outputs in artifacts/")


[6] Scanning test statistic over ctGRe ...


Scanning θ:   0%|          | 0/81 [00:00<?, ?it/s]


AttributeError: 'CalibratedParametric' object has no attribute 'log_likelihood_ratio'